# Stage 2 — Build Network Data

對應範例的 Notebook 1，產生 18 個分析單位（14 系所 + 3 組 + 1 全體）的網路資料表。

**不使用 `pinyin` 與 `googletrans`**：預處理階段已完成確定性雙語標準化，引入即時翻譯會破壞可重現性。

產出：

- `data/network-analysis/keyword_idf.csv`
- `data/network-analysis/person_identity_map.csv`
- `data/network-analysis/units_manifest.json`
- `data/network-analysis/{by-school,by-group,overall}/<unit>/{professor_nodes,keyword_nodes,prof_keyword_edges}.csv`

In [1]:
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "data-preprocessing").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

PRE_DIR = ROOT / "data" / "data-preprocessing"
NET_DIR = ROOT / "data" / "network-analysis"
NET_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = {
    "fin-and-econ": ["nccu-fin", "nccu-mb", "nthu-qf", "ntu-econ", "ntu-fin", "nycu-info-mag-fin"],
    "ds": ["ncku-stats-ds", "ntu-comm-engin-ds-network", "ntu-ds"],
    "cs": ["ncku-cs-ie", "nthu-cs", "ntu-cs-ie", "ntu-im", "nycu-cs"],
}
SCHOOL_TO_GROUP = {s: g for g, ss in GROUPS.items() for s in ss}
SCHOOLS = sorted(SCHOOL_TO_GROUP)

with (PRE_DIR / "faculty_standardized.json").open(encoding="utf-8") as fh:
    FACULTY = json.load(fh)

TAXONOMY = pd.read_csv(NET_DIR / "research_taxonomy.csv", encoding="utf-8-sig")
KW2L2 = dict(zip(TAXONOMY["keyword_en"], TAXONOMY["parent_l2"]))
KW2L1 = dict(zip(TAXONOMY["keyword_en"], TAXONOMY["parent_l1"]))

print(f"faculty records : {len(FACULTY)}")
print(f"taxonomy rows   : {len(TAXONOMY)}")
print(f"schools         : {len(SCHOOLS)}")

faculty records : 741
taxonomy rows   : 170
schools         : 14


## 1. 全體 IDF 權重

```
IDF(k) = ln(1 + N_all / n_k)      N_all = 662（有標準化研究領域的教授紀錄數）
```

以**全體紀錄**計算一次，18 個分析單位共用，確保跨單位邊權重可比較。

In [2]:
records_with_kw = [r for r in FACULTY if r.get("research_interests_standardized_en")]
N_ALL = len(records_with_kw)

kw_holder_counts = Counter()
for rec in records_with_kw:
    for kw in set(rec["research_interests_standardized_en"]):
        kw_holder_counts[kw] += 1

idf_rows = []
for kw, n_k in kw_holder_counts.items():
    idf_rows.append(
        {
            "keyword_en": kw,
            "parent_l2": KW2L2[kw],
            "parent_l1": KW2L1[kw],
            "n_faculty_overall": n_k,
            "idf": math.log(1 + N_ALL / n_k),
        }
    )
idf_df = pd.DataFrame(idf_rows).sort_values("n_faculty_overall", ascending=False).reset_index(drop=True)
idf_df.to_csv(NET_DIR / "keyword_idf.csv", index=False, encoding="utf-8-sig")

IDF = dict(zip(idf_df["keyword_en"], idf_df["idf"]))
N_OVERALL = dict(zip(idf_df["keyword_en"], idf_df["n_faculty_overall"]))

print(f"N_all = {N_ALL}")
print(idf_df.head(5).to_string(index=False))
print("...")
print(idf_df.tail(3).to_string(index=False))

N_all = 662
             keyword_en                                   parent_l2                                     parent_l1  n_faculty_overall      idf
Artificial Intelligence Artificial Intelligence & Generative Models                         AI & Machine Learning                109 1.956340
       Machine Learning            Machine Learning & Deep Learning                         AI & Machine Learning                 94 2.084747
     Multimedia Systems               Graphics & Multimedia Systems Vision, Multimedia & Human-Centered Computing                 57 2.534810
        Computer Vision  Computer Vision, Image & Signal Processing Vision, Multimedia & Human-Centered Computing                 54 2.584696
          Deep Learning            Machine Learning & Deep Learning                         AI & Machine Learning                 54 2.584696
...
           keyword_en                                      parent_l2                                 parent_l1  n_faculty_overall   

## 2. 跨校去重（D3）

0. **人工覆核結果優先**：`MANUAL_SAME_PERSON` / `MANUAL_DIFFERENT_PERSON` 為逐筆人工確認的裁定，
   覆蓋以下所有自動規則。
1. `person_id` 優先以正規化 email（小寫、去空白）判定 —— 共用任一 email 的紀錄視為同一人。
2. 無 email 時以 `professor_name`（非空中文名）比對，並要求 `name_en` 正規化後相容或其一為空。
3. 同名但 email 不同 → 視為不同人，於 `person_identity_map.csv` 標記
   `status = name_collision_kept_separate` 供人工覆核。
4. 合併後 `research_interests_standardized_en` 取**聯集去重**，並保留 `keyword_source_schools`。
5. `school_ids` / `faculty_ids` / `titles` 轉為多值屬性，`n_appointments` 記錄合聘數。

In [3]:
CJK_RE = re.compile(r"[㐀-䶿一-鿿]")


def norm_email(value: str) -> str:
    """小寫、去空白，並移除 HTML `&nbsp;` 殘留造成的 `nbsp` 前綴。

    例：`f522a412fdd1552e@redacted.invalid` -> `f522a412fdd1552e@redacted.invalid`，
    這是同一個人在兩個系所網頁上被寫成兩個不同 email 的主因之一。
    """
    text = unicodedata.normalize("NFKC", str(value)).strip().lower()
    text = re.sub(r"^(?:&nbsp;|nbsp)+", "", text)
    return text.strip()


def norm_name_en(value) -> str:
    if not value:
        return ""
    text = unicodedata.normalize("NFKC", str(value)).lower()
    text = re.sub(r"[^a-z]+", " ", text)
    return " ".join(sorted(text.split()))


def norm_name_zh(value) -> str:
    if not value:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", "", text)
    return text


class DisjointSet:
    def __init__(self) -> None:
        self.parent: dict[str, str] = {}

    def find(self, x: str) -> str:
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


dsu = DisjointSet()
for rec in FACULTY:
    dsu.find(rec["faculty_id"])

BY_FID = {r["faculty_id"]: r for r in FACULTY}

# --- rule 0: 人工覆核裁定（優先於所有自動規則） ----------------------------
# 自動規則只能看到 email 與姓名字串；同一位教授在兩校使用不同網域的信箱時，規則 3 會保守地
# 拆成兩個人並標記待覆核。下表是逐筆人工確認後的裁定，直接覆蓋自動判定結果。
MANUAL_SAME_PERSON: list[tuple[str, list[str]]] = [
    ("李家岩 Chia-Yen Lee", ["ncku-cs-ie-李家岩-10568c76d0", "ntu-im-李家岩-6ba583b1b6"]),
    ("連震杰 Jenn-Jier James Lien", ["ncku-cs-ie-連震杰-f316933353", "ncku-stats-ds-連震杰-07850768ed"]),
    ("林桂傑 Kwei-Jay Lin", ["nthu-cs-林桂傑-3eab7348de", "ntu-cs-ie-林桂傑-b717b8860b"]),
    ("俞明德 Min-Teh Yu", ["nthu-qf-俞明德-5bc6444834", "nycu-info-mag-fin-俞明德-560565dfd3"]),
    ("李琳山 Lin-Shan Lee", ["ntu-comm-engin-ds-network-李琳山-193d6b842c", "ntu-cs-ie-lslee-906cf8b461"]),
]
MANUAL_DIFFERENT_PERSON: list[tuple[str, list[str]]] = [
    ("陳耀宗 Yaw-Chung Chen / Yao-Tsung Chen",
     ["nycu-cs-yawchungchen-fcbdbf6aa9", "nycu-info-mag-fin-陳耀宗-5cb5b0b06f"]),
]

_manual_ids = [f for _, fids in MANUAL_SAME_PERSON + MANUAL_DIFFERENT_PERSON for f in fids]
_missing = [f for f in _manual_ids if f not in BY_FID]
assert not _missing, f"人工裁定中的 faculty_id 不存在：{_missing}"

MANUAL_SAME_FIDS = {f for _, fids in MANUAL_SAME_PERSON for f in fids}
MANUAL_DIFFERENT_FIDS = {f for _, fids in MANUAL_DIFFERENT_PERSON for f in fids}
FORBIDDEN_PAIRS: set[frozenset[str]] = {
    frozenset((a, b))
    for _, fids in MANUAL_DIFFERENT_PERSON
    for i, a in enumerate(fids)
    for b in fids[i + 1:]
}

manual_decisions = pd.DataFrame(
    [
        {"decision": "same_person", "label": label, "faculty_ids": " | ".join(fids),
         "school_ids": " | ".join(BY_FID[f]["school_id"] for f in fids)}
        for label, fids in MANUAL_SAME_PERSON
    ]
    + [
        {"decision": "different_person", "label": label, "faculty_ids": " | ".join(fids),
         "school_ids": " | ".join(BY_FID[f]["school_id"] for f in fids)}
        for label, fids in MANUAL_DIFFERENT_PERSON
    ]
)
manual_decisions.to_csv(NET_DIR / "manual_identity_decisions.csv", index=False, encoding="utf-8-sig")
print("人工覆核裁定：")
print(manual_decisions[["decision", "label", "school_ids"]].to_string(index=False))

# --- rule 1: shared personal email ---------------------------------------
# 系所頁面常把「系辦公室 / 聯絡窗口」信箱重複掛在多位教授上（例：`1a9bfd52a6de752e@redacted.invalid`
# 出現在 ncku-cs-ie 的 62 筆紀錄）。這種信箱不是個人識別碼。
# 判定規則：同一個信箱若在**同一個 school_id 內出現 2 筆以上**，即視為 shared contact email
# 並排除於身分比對之外——同一份系所名單裡同一個人不應出現兩次。
email_index: dict[str, list[str]] = defaultdict(list)
for rec in FACULTY:
    for em in rec.get("emails") or []:
        em = norm_email(em)
        if em:
            email_index[em].append(rec["faculty_id"])

SHARED_CONTACT_EMAILS = {
    em
    for em, fids in email_index.items()
    if Counter(BY_FID[f]["school_id"] for f in fids).most_common(1)[0][1] > 1
}
shared_email_df = pd.DataFrame(
    [
        {
            "email": em,
            "n_records": len(email_index[em]),
            "school_ids": " | ".join(sorted({BY_FID[f]["school_id"] for f in email_index[em]})),
        }
        for em in sorted(SHARED_CONTACT_EMAILS)
    ]
)
print("排除的 shared contact email：")
print(shared_email_df.to_string(index=False) if len(shared_email_df) else "  （無）")
shared_email_df.to_csv(NET_DIR / "shared_contact_emails.csv", index=False, encoding="utf-8-sig")

for em, fids in email_index.items():
    if em in SHARED_CONTACT_EMAILS:
        continue
    for fid in fids[1:]:
        dsu.union(fids[0], fid)

def personal_emails(rec) -> list[str]:
    return sorted(
        {
            norm_email(e)
            for e in (rec.get("emails") or [])
            if norm_email(e) and norm_email(e) not in SHARED_CONTACT_EMAILS
        }
    )

has_email = {fid: bool(personal_emails(r)) for fid, r in BY_FID.items()}

# 套用人工確認的合併（規則 0），在姓名比對之前先生效
for _, fids in MANUAL_SAME_PERSON:
    for fid in fids[1:]:
        dsu.union(fids[0], fid)


# --- rule 2: no email -> match on Chinese professor_name + compatible name_en
name_index: dict[str, list[str]] = defaultdict(list)
for rec in FACULTY:
    zh = norm_name_zh(rec.get("professor_name"))
    if zh and CJK_RE.search(zh):
        name_index[zh].append(rec["faculty_id"])

name_merge_notes: dict[str, str] = {}
for zh, fids in name_index.items():
    if len(fids) < 2:
        continue
    emailless = [f for f in fids if not has_email[f]]
    if not emailless:
        continue
    for fid in emailless:
        en_a = norm_name_en(BY_FID[fid].get("name_en"))
        targets = []
        for other in fids:
            if other == fid or dsu.find(other) == dsu.find(fid):
                continue
            if frozenset((fid, other)) in FORBIDDEN_PAIRS:
                continue
            en_b = norm_name_en(BY_FID[other].get("name_en"))
            if not en_a or not en_b or en_a == en_b:
                targets.append(other)
        clusters = {dsu.find(t) for t in targets}
        if len(clusters) == 1:
            dsu.union(fid, targets[0])
            name_merge_notes[fid] = "merged_by_name_no_email"
        elif len(clusters) > 1:
            name_merge_notes[fid] = "ambiguous_name_kept_separate"

# --- rule 3: same Chinese name spread over >1 person cluster -> flag -------
collision_fids: set[str] = set()
for zh, fids in name_index.items():
    clusters = {dsu.find(f) for f in fids}
    if len(clusters) > 1:
        collision_fids.update(fids)

person_of = {fid: dsu.find(fid) for fid in BY_FID}
person_members: dict[str, list[str]] = defaultdict(list)
for fid, pid in person_of.items():
    person_members[pid].append(fid)

identity_rows = []
for fid, rec in BY_FID.items():
    pid = person_of[fid]
    members = person_members[pid]
    schools = sorted({BY_FID[m]["school_id"] for m in members})
    if fid in MANUAL_SAME_FIDS:
        status = "manually_confirmed_same_person"
    elif fid in MANUAL_DIFFERENT_FIDS:
        status = "manually_confirmed_different_person"
    elif fid in collision_fids:
        status = "name_collision_kept_separate"
    elif len(members) > 1:
        status = "merged_multi_record"
    else:
        status = "unique"
    identity_rows.append(
        {
            "faculty_id": fid,
            "person_id": pid,
            "school_id": rec["school_id"],
            "professor_name": rec.get("professor_name"),
            "name_en": rec.get("name_en"),
            "emails": " | ".join(personal_emails(rec)),
            "n_records_in_person": len(members),
            "n_schools_in_person": len(schools),
            "person_school_ids": " | ".join(schools),
            "status": status,
            "match_note": (
                "manual_review"
                if fid in MANUAL_SAME_FIDS or fid in MANUAL_DIFFERENT_FIDS
                else name_merge_notes.get(fid, "email_or_singleton")
            ),
        }
    )

identity_df = pd.DataFrame(identity_rows).sort_values(["person_id", "school_id"]).reset_index(drop=True)
identity_df.to_csv(NET_DIR / "person_identity_map.csv", index=False, encoding="utf-8-sig")

n_persons = identity_df["person_id"].nunique()
multi_school = identity_df[identity_df["n_schools_in_person"] > 1]
print(f"records                       : {len(identity_df)}")
print(f"distinct persons              : {n_persons}")
print(f"cross-school duplicate persons: {multi_school['person_id'].nunique()}  ({len(multi_school)} records)")
print(f"manually merged (rule 0)      : {len(MANUAL_SAME_FIDS)} records in {len(MANUAL_SAME_PERSON)} persons")
print(f"manually kept separate        : {len(MANUAL_DIFFERENT_FIDS)} records")
print(f"name_collision_kept_separate  : {(identity_df['status'] == 'name_collision_kept_separate').sum()} records（待覆核）")
print()
print("每組組內跨校重複人數：")
for gname, gschools in GROUPS.items():
    sub = identity_df[identity_df["school_id"].isin(gschools)]
    dup = sub.groupby("person_id")["school_id"].nunique()
    dup = dup[dup > 1]
    print(f"  {gname:14s} records={len(sub):4d}  dup_persons={len(dup):3d}  deduped={sub['person_id'].nunique():4d}")

人工覆核裁定：
        decision                               label                            school_ids
     same_person                    李家岩 Chia-Yen Lee                   ncku-cs-ie | ntu-im
     same_person            連震杰 Jenn-Jier James Lien            ncku-cs-ie | ncku-stats-ds
     same_person                    林桂傑 Kwei-Jay Lin                   nthu-cs | ntu-cs-ie
     same_person                      俞明德 Min-Teh Yu           nthu-qf | nycu-info-mag-fin
     same_person                    李琳山 Lin-Shan Lee ntu-comm-engin-ds-network | ntu-cs-ie
different_person 陳耀宗 Yaw-Chung Chen / Yao-Tsung Chen           nycu-cs | nycu-info-mag-fin
排除的 shared contact email：
                    email  n_records school_ids
1a9bfd52a6de752e@redacted.invalid         62 ncku-cs-ie
       95f6910acde21198@redacted.invalid         43    ntu-fin
        2432aaca30fd1524@redacted.invalid          2   ntu-econ
    0c8553022a7a2824@redacted.invalid          3   ntu-econ
   4ef3a5beaca61a4a@redacted.invalid  

## 3. 建立 person 層級檔案（研究領域取聯集）

In [4]:
PERSON: dict[str, dict] = {}
for pid, members in person_members.items():
    recs = [BY_FID[m] for m in members]
    recs.sort(key=lambda r: r["faculty_id"])
    kw_sources: dict[str, set[str]] = defaultdict(set)
    for r in recs:
        for kw in r.get("research_interests_standardized_en") or []:
            kw_sources[kw].add(r["school_id"])
    phd = ""
    for r in recs:
        for edu in r.get("education_structured") or []:
            if (edu.get("degree") or "").strip().lower() in {"phd", "ph.d.", "doctorate"} and edu.get("institution"):
                phd = edu["institution"]
                break
        if phd:
            break
    name = next((r.get("professor_name") for r in recs if r.get("professor_name")), "")
    name_en = next((r.get("name_en") for r in recs if r.get("name_en")), "")
    PERSON[pid] = {
        "person_id": pid,
        "faculty_ids": [r["faculty_id"] for r in recs],
        "school_ids": sorted({r["school_id"] for r in recs}),
        "professor_name": name,
        "name_en": name_en,
        "role_groups": sorted({g for r in recs for g in (r.get("role_groups") or [])}),
        "titles": sorted({t for r in recs for t in (r.get("titles_standardized_en") or [])}),
        "keywords": sorted(kw_sources),
        "keyword_source_schools": {k: sorted(v) for k, v in kw_sources.items()},
        "phd_institution": phd,
        "n_appointments": len({r["school_id"] for r in recs}),
    }

print(f"person records built: {len(PERSON)}")
print(f"persons with keywords: {sum(1 for p in PERSON.values() if p['keywords'])}")

person records built: 701
persons with keywords: 623


## 4. 產生 18 個分析單位的三份表

- **系所層級**：不去重，`node_id = faculty_id`。
- **組層級 / 全體層級**：以 `person_id` 去重，`node_id = person_id`，研究領域取聯集。
- 無研究領域者不進入網路，但排除人數記錄於 `units_manifest.json`。

`prof_keyword_edges.csv` 的 `source` 欄位 = 該領域來自哪一個（或哪些）系所網頁（D3 規則 4 的
`keyword_source_schools`）；系所層級恆為該系所本身。

In [5]:
def clean_multi(values) -> str:
    return " | ".join(values)


def build_unit(unit_key: str, level: str, school_ids: list[str], dedup: bool) -> dict:
    """回傳 unit manifest，並寫出三份 CSV。"""
    if level == "school":
        out_dir = NET_DIR / "by-school" / unit_key
    elif level == "group":
        out_dir = NET_DIR / "by-group" / unit_key
    else:
        out_dir = NET_DIR / "overall"
    out_dir.mkdir(parents=True, exist_ok=True)

    unit_records = [r for r in FACULTY if r["school_id"] in school_ids]
    n_records = len(unit_records)

    if dedup:
        pids = sorted({person_of[r["faculty_id"]] for r in unit_records})
        entities = []
        for pid in pids:
            p = PERSON[pid]
            in_unit_schools = [s for s in p["school_ids"] if s in school_ids]
            kw_src = {
                k: [s for s in v if s in school_ids]
                for k, v in p["keyword_source_schools"].items()
            }
            kw_src = {k: v for k, v in kw_src.items() if v}
            entities.append(
                {
                    "node_id": pid,
                    "professor_name": p["professor_name"],
                    "name_en": p["name_en"],
                    "school_ids": in_unit_schools,
                    "role_groups": p["role_groups"],
                    "keywords": sorted(kw_src),
                    "keyword_source_schools": kw_src,
                    "phd_institution": p["phd_institution"],
                    "n_appointments": len(in_unit_schools),
                }
            )
    else:
        entities = []
        for r in unit_records:
            kws = sorted(set(r.get("research_interests_standardized_en") or []))
            phd = ""
            for edu in r.get("education_structured") or []:
                if (edu.get("degree") or "").strip().lower() in {"phd", "ph.d.", "doctorate"} and edu.get("institution"):
                    phd = edu["institution"]
                    break
            entities.append(
                {
                    "node_id": r["faculty_id"],
                    "professor_name": r.get("professor_name") or "",
                    "name_en": r.get("name_en") or "",
                    "school_ids": [r["school_id"]],
                    "role_groups": sorted(r.get("role_groups") or []),
                    "keywords": kws,
                    "keyword_source_schools": {k: [r["school_id"]] for k in kws},
                    "phd_institution": phd,
                    "n_appointments": len(PERSON[person_of[r["faculty_id"]]]["school_ids"]),
                }
            )

    n_entities = len(entities)
    with_kw = [e for e in entities if e["keywords"]]
    n_excluded = n_entities - len(with_kw)

    prof_rows = [
        {
            "node_id": e["node_id"],
            "professor_name": e["professor_name"],
            "name_en": e["name_en"],
            "school_ids": clean_multi(e["school_ids"]),
            "role_groups": clean_multi(e["role_groups"]),
            "n_keywords": len(e["keywords"]),
            "phd_institution": e["phd_institution"],
            "n_appointments": e["n_appointments"],
        }
        for e in with_kw
    ]
    prof_df = pd.DataFrame(prof_rows).sort_values("node_id").reset_index(drop=True)

    edge_rows = [
        {"node_id": e["node_id"], "keyword_en": kw, "source": clean_multi(e["keyword_source_schools"][kw])}
        for e in with_kw
        for kw in e["keywords"]
    ]
    edge_df = pd.DataFrame(edge_rows).sort_values(["node_id", "keyword_en"]).reset_index(drop=True)

    kw_counts = Counter(edge_df["keyword_en"])
    kw_rows = [
        {
            "keyword_en": kw,
            "parent_l2": KW2L2[kw],
            "parent_l1": KW2L1[kw],
            "n_faculty_in_unit": cnt,
            "n_faculty_overall": N_OVERALL[kw],
            "idf": IDF[kw],
        }
        for kw, cnt in kw_counts.items()
    ]
    kw_df = (
        pd.DataFrame(kw_rows)
        .sort_values(["n_faculty_in_unit", "keyword_en"], ascending=[False, True])
        .reset_index(drop=True)
    )

    prof_df.to_csv(out_dir / "professor_nodes.csv", index=False, encoding="utf-8-sig")
    kw_df.to_csv(out_dir / "keyword_nodes.csv", index=False, encoding="utf-8-sig")
    edge_df.to_csv(out_dir / "prof_keyword_edges.csv", index=False, encoding="utf-8-sig")

    if dedup:
        nodedup_rows = []
        for r in unit_records:
            kws = sorted(set(r.get("research_interests_standardized_en") or []))
            if not kws:
                continue
            nodedup_rows.append(
                {
                    "node_id": r["faculty_id"],
                    "person_id": person_of[r["faculty_id"]],
                    "professor_name": r.get("professor_name") or "",
                    "name_en": r.get("name_en") or "",
                    "school_ids": r["school_id"],
                    "role_groups": clean_multi(sorted(r.get("role_groups") or [])),
                    "n_keywords": len(kws),
                    "n_appointments": len(PERSON[person_of[r["faculty_id"]]]["school_ids"]),
                }
            )
        pd.DataFrame(nodedup_rows).sort_values("node_id").to_csv(
            out_dir / "professor_nodes_nodedup.csv", index=False, encoding="utf-8-sig"
        )

    dup_persons = 0
    if dedup:
        cnt = Counter(person_of[r["faculty_id"]] for r in unit_records)
        dup_persons = sum(1 for v in cnt.values() if v > 1)

    return {
        "unit": unit_key,
        "level": level,
        "school_ids": school_ids,
        "dedup": dedup,
        "n_records": n_records,
        "n_entities": n_entities,
        "n_duplicate_persons_removed": n_records - n_entities,
        "n_duplicate_persons": dup_persons,
        "n_in_network": len(with_kw),
        "n_excluded_no_keywords": n_excluded,
        "coverage_pct": round(100 * len(with_kw) / n_entities, 1) if n_entities else 0.0,
        "n_keywords": int(kw_df.shape[0]),
        "n_prof_keyword_edges": int(edge_df.shape[0]),
        "out_dir": str(out_dir.relative_to(ROOT)).replace("\\", "/"),
    }


manifest = []
for school in SCHOOLS:
    manifest.append(build_unit(school, "school", [school], dedup=False))
for gname, gschools in GROUPS.items():
    manifest.append(build_unit(gname, "group", gschools, dedup=True))
manifest.append(build_unit("overall", "overall", SCHOOLS, dedup=True))

manifest_df = pd.DataFrame(manifest)
print(
    manifest_df[
        [
            "unit",
            "level",
            "n_records",
            "n_entities",
            "n_duplicate_persons_removed",
            "n_in_network",
            "n_excluded_no_keywords",
            "coverage_pct",
            "n_keywords",
            "n_prof_keyword_edges",
        ]
    ].to_string(index=False)
)

with (NET_DIR / "units_manifest.json").open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

                     unit   level  n_records  n_entities  n_duplicate_persons_removed  n_in_network  n_excluded_no_keywords  coverage_pct  n_keywords  n_prof_keyword_edges
                 nccu-fin  school         23          23                            0            23                       0         100.0          24                    62
                  nccu-mb  school         22          22                            0            18                       4          81.8          23                    44
               ncku-cs-ie  school         62          62                            0            51                      11          82.3          66                   189
            ncku-stats-ds  school         47          47                            0            39                       8          83.0          44                   124
                  nthu-cs  school         81          81                            0            76                       5          93.8   

## 5. 驗證

In [6]:
BASELINE_COVERAGE = {
    "nccu-fin": (23, 23),
    "nccu-mb": (22, 18),
    "ncku-cs-ie": (62, 51),
    "ncku-stats-ds": (47, 39),
    "nthu-cs": (81, 76),
    "nthu-qf": (19, 17),
    "ntu-comm-engin-ds-network": (25, 25),
    "ntu-cs-ie": (90, 88),
    "ntu-ds": (69, 59),
    "ntu-econ": (49, 44),
    "ntu-fin": (43, 32),
    "ntu-im": (31, 30),
    "nycu-cs": (152, 141),
    "nycu-info-mag-fin": (28, 19),
}

checks: list[tuple[str, bool, str]] = []

bad = []
for row in manifest:
    if row["level"] != "school":
        continue
    exp_total, exp_net = BASELINE_COVERAGE[row["unit"]]
    if (row["n_entities"], row["n_in_network"]) != (exp_total, exp_net):
        bad.append((row["unit"], row["n_entities"], row["n_in_network"], exp_total, exp_net))
checks.append(("各系所節點數與計畫 §1 覆蓋率表一致", not bad, str(bad)))

bad = []
for row in manifest:
    if row["level"] not in {"group", "overall"}:
        continue
    if row["n_entities"] != row["n_records"] - row["n_duplicate_persons_removed"]:
        bad.append(row["unit"])
checks.append(("去重後人數 = 原紀錄數 − 重複紀錄數", not bad, str(bad)))

all_kw = set(TAXONOMY["keyword_en"])
bad = []
for row in manifest:
    edges = pd.read_csv(ROOT / row["out_dir"] / "prof_keyword_edges.csv", encoding="utf-8-sig")
    unknown = set(edges["keyword_en"]) - all_kw
    if unknown:
        bad.append((row["unit"], sorted(unknown)[:5]))
checks.append(("邊表中所有 keyword 存在於 research_taxonomy.csv", not bad, str(bad)))

bad = []
for row in manifest:
    nodes = pd.read_csv(ROOT / row["out_dir"] / "professor_nodes.csv", encoding="utf-8-sig")
    edges = pd.read_csv(ROOT / row["out_dir"] / "prof_keyword_edges.csv", encoding="utf-8-sig")
    if set(edges["node_id"]) != set(nodes["node_id"]):
        bad.append(row["unit"])
    if nodes["node_id"].duplicated().any():
        bad.append(row["unit"] + ":dup_node_id")
checks.append(("節點表與邊表的 node_id 一致且無重複", not bad, str(bad)))

overall_row = next(r for r in manifest if r["level"] == "overall")
checks.append(
    (
        f"全體層級研究領域數 = 170（實際 {overall_row['n_keywords']}）",
        overall_row["n_keywords"] == 170,
        "",
    )
)

for label, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  {detail}" if not ok else ""))

assert all(ok for _, ok, _ in checks), "Stage 2 驗證未通過"
print("\nStage 2 完成。")

[PASS] 各系所節點數與計畫 §1 覆蓋率表一致
[PASS] 去重後人數 = 原紀錄數 − 重複紀錄數
[PASS] 邊表中所有 keyword 存在於 research_taxonomy.csv
[PASS] 節點表與邊表的 node_id 一致且無重複
[PASS] 全體層級研究領域數 = 170（實際 170）

Stage 2 完成。
